In [ ]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.datasets import make_classification 
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.feature_selection import SelectKBest, SelectFpr, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import SnowballStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
import re
import string
from bs4 import BeautifulSoup

import nltk
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [ ]:
rows = df.loc[df.duplicated(subset=['article', 'title'])]
duplicated = df.loc[df.duplicated(subset=['title','article'])]

df.dropna(inplace=True)

In [ ]:
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

for i, article in zip(df.index, df['article']):
    df.loc[i, 'article'] = clean_text(article)

In [ ]:
display(df)

In [ ]:
mask = ((df['label'] == 0) | (df['label'] == 5))
subset = df[mask]

In [ ]:
df = subset

In [ ]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
text = df['title'] + ' ' + df['title'] + ' ' + df['article']
df = pd.concat([df, text], axis=1)

#rinomino le colonne e droppo title e article
df.rename(columns={0: 'text'}, inplace=True)
df.drop(['title', 'article'], inplace=True, axis=1)
df.columns

In [ ]:
stemmer = SnowballStemmer('english')
def stemming_tokenizer(text):
    # Separa le parole
    words = text.split()
    # Applica lo stemmer a ogni parola e riunisci
    # (È qui che avviene la magia: "running" -> "run")
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

df['text'] = df['text'].apply(stemming_tokenizer)

In [ ]:
#y = df['label']
#df.drop('label', inplace=True, axis=1)

In [ ]:
df.drop('timestamp', inplace=True, axis=1)

In [ ]:
df.drop('Id', inplace=True, axis=1)

In [ ]:
df.drop('page_rank', inplace=True, axis=1)

In [ ]:
stop_words = list(ENGLISH_STOP_WORDS) + [
    # --- Rumore HTML e Scraping ---
    'http', 'https', 'src', 'href', 'img', 'alignleft', 'alignright', 
    'height', 'width', 'border', 'clearall', 'borderimga', 'br', 'class',
    
    # --- Agenzie Stampa e Formattazione News ---
    'reuters', 'ap', 'afp', 'pa', 'upi', 'press', 'association',
    'new', 'york', 'london', 'washington', # Città ricorrenti nelle dateline
    'said', 'says', 'told', 'reported', 'added', # Verbi di reportistica
    
    # --- Temporali Generici (inutili per classificare l'argomento) ---
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'yesterday', 'today', 'tomorrow', 'year', 'years', 'week', 'month', 'day',
    'time', 'daily', 'date', 'attack', 'baghdad', 'bomb', 'bush', 'gaza', 'iran', 'iraq',
    'iraqi', 'israel', 'kerry', 'palestinian','british' ,'countri', 'court','death', 'democrat' ,'elect' ,'forc',
 'govern' ,'group' ,'kill', 'leader', 'militari', 'minist', 'nation', 'offici',
 'peopl', 'plan', 'polic', 'presid', 'prime', 'prime minist', 'reuters', 'secur',
 'state', 'talk' ,'unit', 'vote', 'war' ,'world']

stemmed_stop_words = [stemmer.stem(word) for word in stop_words]

In [ ]:
df

In [ ]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [ ]:
encoder = OneHotEncoder(min_frequency=100, handle_unknown='infrequent_if_exist')

scaler = MinMaxScaler()

text_pipeline_selector = Pipeline([
    ('vect', TfidfVectorizer(
        stop_words=stemmed_stop_words,
        max_df = 0.8,
        max_features = 30000,
        ngram_range=(1, 2),
        use_idf=False
    )),
    ('selector', SelectFpr(score_func=chi2, alpha=0.001)),
    ('selector2', SelectFpr(score_func=f_classif, alpha=0.001))
])

vectorizer = TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        min_df=3)

vectorizer_title = TfidfVectorizer(
        max_features=3000,
        stop_words=stemmed_stop_words,
        ngram_range=(1, 2),
        min_df=3)

preprocessor = ColumnTransformer(transformers=[
    ('text', text_pipeline_selector, 'text'),
    ('source', encoder, ['source'])
], remainder='drop')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y, shuffle=True)

In [ ]:
X_train_final = preprocessor.fit_transform(X_train, y_train)
X_train_final = preprocessor.transform(X_test)

In [ ]:
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix

weights = compute_sample_weight(class_weight='balanced', y=y_train)


clf = LogisticRegression(
    C=1,                       
    class_weight='balanced',    
    solver='liblinear',        
    penalty='l2',           
    max_iter=1000,              
    random_state=42
)
clf2 = RandomForestClassifier(
    random_state= 42,
    n_estimators=1500,
    max_depth=10,
    class_weight='balanced',
    n_jobs = -1
)

clf3 = GradientBoostingClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    random_state=42,
    warm_start=True,
)

svm = LinearSVC(
        C=1,
        class_weight='balanced',
        max_iter=2000,
        random_state=42,
        penalty='l1'
)

clf5 = XGBClassifier(
        n_estimators=1000,      
        learning_rate=0.05,     
        max_depth=6,             
        min_child_weight=1,
        gamma=0.1,           
        subsample=0.8,           
        colsample_bytree=0.8,     
        n_jobs=-1,               
        random_state=42,
        eval_metric='mlogloss'  
    )

clf_svm_prob = CalibratedClassifierCV(svm, method='sigmoid')
clf_nb = MultinomialNB()

voting_clf = VotingClassifier(
    estimators=[ 
        ('svm', clf_svm_prob), 
        ('xgb', clf5)
    ],
    voting='soft', weights=[2,1], verbose=True
)

model = svm

model.fit(X_train_final, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
# 1. Configurazione XGBoost: GLI DIAMO TUTTA LA POTENZA
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

clf_xgb = XGBClassifier(
    tree_method="hist", 
    n_estimators=1000, 
    max_depth=6, 
    learning_rate=0.05,
    n_jobs=-1,          # <--- LUI DEVE USARE TUTTI I CORE (100% CPU)
    random_state=42
)

# 2. Configurazione SVM
clf_svm = SGDClassifier(
    loss='modified_huber', 
    penalty='l2', 
    alpha=1e-4, 
    random_state=42, 
    max_iter=1000, 
    tol=1e-3,
    class_weight='balanced',
    n_jobs=-1           # Anche lui può usare tutto quando tocca a lui
)

# 3. Configurazione Voting: NON PARALLELIZZARE QUI!
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', clf_xgb), 
        ('svm', clf_svm)
    ],
    voting='soft',
    weights=[1, 1],
    n_jobs=1  # <--- FONDAMENTALE: 1 significa "Esegui uno dopo l'altro"
)

# Definiamo cosa vogliamo testare
param_dist = {
    # Parametri per XGBoost
    'xgb__n_estimators': [500, 1000, 1500],
    'xgb__max_depth': [3, 6, 9],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.7, 0.8, 0.9],
    
    # Parametri per SVM (SGDClassifier)
    'svm__alpha': np.logspace(-5, -2, 4),
    'svm__penalty': ['l2', 'elasticnet'],
    
    # Pesi del Voting
    'weights': [[1, 1], [2, 1], [1, 2]]
}

In [ ]:
# Creiamo il tuner
random_search = RandomizedSearchCV(
    estimator=voting_clf,
    param_distributions=param_dist,
    n_iter=20,           # Numero di combinazioni casuali da testare
    cv=3,                # Cross-validation a 3 fold (veloce per iniziare)
    scoring='f1_weighted',
    n_jobs=-1,           # <--- Fondamentale: usa tutti i core del Mac per i diversi test
    verbose=3,
    random_state=42
)

# Avvio del tuning
print("Inizio Hyperparameter Tuning (questo richiederà tempo)...")
X_train_transformed = preprocessor_custom.fit_transform(X_train)
random_search.fit(X_train_transformed, y_train)

# Risultati
print(f"Migliori parametri trovati: {random_search.best_params_}")
print(f"Miglior Score: {random_search.best_score_}")

# Usa il miglior modello trovato
best_voting_clf = random_search.best_estimator_

In [ ]:
print("📊 Valutazione...")
y_pred = voting_clf.predict(X_test_transformed)
print(classification_report(y_test, y_pred))

In [ ]:
#weights = compute_sample_weight(class_weight='balanced', y=y_train)
## Definisci il modello XGBoost
#clf = XGBClassifier(
#    n_estimators=1000,      # Numero di alberi
#    learning_rate=0.1,     # Impara piano per essere preciso
#    max_depth=6,            # Profondità degli alberi
#    subsample=0.8,          # Usa solo l'80% dei dati per ogni albero (evita overfitting)
#    colsample_bytree=0.8,   # Usa solo l'80% delle feature per albero
#    n_jobs=-1,              # Usa tutti i core
#    random_state=42,
#    tree_method="hist"      # Molto più veloce su dataset grandi
#    # Non serve class_weight='balanced', XGBoost gestisce meglio.
#    # Se vuoi forzarlo, usa scale_pos_weight ma è complesso per multiclasse.
#)
#
#X_train = preprocessor_custom.fit_transform(X_train)
#
## Fit
#clf.fit(X_train, y_train, sample_weight=weights)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = clf.predict(X_test)
#
#
#print(classification_report(y_test, y_pred))

In [ ]:
#X_train = preprocessor_custom.fit_transform(X_train)
#clf = LogisticRegression(random_state=RANDOM_SEED,C=1, penalty='l2', solver='saga', class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = clf.predict(X_test)
#
#print(classification_report(y_test, y_pred))

In [ ]:
y_train_pred = clf.predict(X_train)
print(classification_report(y_train, y_train_pred))
